# evaluate GPU utilisation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
from cmath import phase

from pyparsing import col


def load_gpu_logs(file_dict):
    """
    file_dict example:
      {
        "method_A": "methodA.csv",
        "method_B": "methodB.csv"
      }
    Returns: combined dataframe with column 'method'
    """
    dfs = []
    for method, path in file_dict.items():
        df = pd.read_csv(path)
        df["method"] = method
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)
  
def classify(util, idle_thresh=10, active_thresh=60):
    if util < idle_thresh:
        return "idle"
    elif util < active_thresh:
        return "moderate"
    else:
        return "active"
          
def classify_util(df, column, idle_thresh=10, active_thresh=60):
  """
  Returns percent of time GPU spent idle / moderate / active per method
  """

  df["util_state"] = df[column].apply(classify, args=(idle_thresh, active_thresh))
  summary = df.groupby(['method', 'util_state']).size().rename("count").groupby(level=0).transform(lambda x: round(100 * x / x.sum(), 2)).rename("usage (%)").reset_index()
  return summary

def per_phase_gpu_stats(df):
    """
    Returns avg GPU & memory utilization per phase and per method
    """
    stats = (
        df.groupby(["method", "phase"])
          .agg(
              gpu_util_median=("gpu_util_percent", "median"),
              gpu_util_mean=("gpu_util_percent", "mean"),
          )
          .reset_index()
    )
    stats = stats.round(2)
    return stats

def per_phase_mem_stats(df):
    """
    Returns avg GPU & memory utilization per phase and per method
    """
    stats = (
        df.groupby(["method", "phase"])
          .agg(
              mem_util_median=("mem_util_percent", "median"),
              mem_util_mean=("mem_util_percent", "mean"),
          )
          .reset_index()
    )    
    stats = stats.round(2)
    return stats

def per_runtime_stats(df, recording_interval_sec=0.1):
    """
    Returns GPU time spent per phase and per method
    """
    stats = (
        df.groupby(["method", "phase"])
          .agg(
              freq=("time", "size"),
          )
          .reset_index()
    )
    stats["runtime (s)"] = stats["freq"] * recording_interval_sec
    stats = stats.drop(columns=["freq"])
    total_runtime = stats.groupby("method")["runtime (s)"].transform("sum")
    stats["runtime (%)"] = 100 * stats["runtime (s)"] / total_runtime
    stats = stats.round(2)
    return stats

def agg_per_step(df):
    """
    Returns avg GPU & memory utilization per phase and per method
    """
    df_aggr = (
        df.groupby(["method", "phase", "step"])
          .agg(
              gpu_util_median=("gpu_util_percent", "median"),
              gpu_util_mean=("gpu_util_percent", "mean"),
              mem_util_median=("mem_util_percent", "median"),
              mem_util_mean=("mem_util_percent", "mean"),
          )
          .reset_index()
    )    
    df_aggr = df_aggr.round(2)
    return df_aggr


def plot_gpu_util_overlay(df, val_col):
    """
    Overlays GPU utilization vs step curves per method
    """
    plt.figure(figsize=(10,5))

    for method, d in df.groupby("method"):
        d_sorted = d.sort_values("step")
        plt.plot(d_sorted["step"], d_sorted[val_col], alpha=0.6, label=method)

    plt.xlabel("Step")
    plt.ylabel("GPU Utilization (%)")
    plt.title("GPU Utilization vs Training Step")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_smoothed_util(df, val_col, window=20, device="GPU", calculation='mean'):
    """
    Rolling mean smoothing per method
    """
    plt.figure(figsize=(10,5))

    for method, d in df.groupby("method"):
        d_sorted = d.sort_values("step")
        smooth = d_sorted[val_col].rolling(window=window).mean()
        plt.plot(d_sorted["step"], smooth, label=f"{method} (smoothed)")

    plt.xlabel("Step")
    plt.ylabel(f"Smoothed {device} Utilization (%)")
    plt.title(f"Smoothed {calculation} {device} Utilization (window={window})")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_smoothed_util_per_phase(df, method, val_col, oppacity=0.2, window=20, device="GPU", calculation='mean'):
    """
    Rolling mean smoothing per method and phase
    """
    plt.figure(figsize=(10,5))
    PHASE_COLORS = {
    "init": "#2c22c5",             # light gray   
    "train": "#696993",            # light blue
    "train_idle": "#baaf90",       # light orange
    "validation": "#AC7D82",       # light green
    "validation_idle": "#ABBCC4",  # light pink
}
    
    df = df.where(df["method"]==method).dropna()
    d_sorted = df.sort_values("step")
    smooth = d_sorted[val_col].rolling(window=window).mean()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(d_sorted["step"], smooth, label=f"{method}", linewidth=3, color="black")
    # fill background per phase
    for row in df.itertuples(index=False):
        ax.axvspan(row.step, row.step + 1, alpha=oppacity, color=PHASE_COLORS[row.phase])
    ax.set_xticks(range(0, int(df['step'].max()), 10))
    ax.set_xlabel("Step")
    ax.set_ylabel(f"Smoothed {device} Utilization (%)")
    ax.set_title(f"Smoothed {calculation} {device} Utilization per Phase (window={window})")
    # --- legends ---
    phase_handles = [Patch(facecolor=color, edgecolor="none", label=phase) for phase, color in PHASE_COLORS.items()]
    line_handles, line_labels = ax.get_legend_handles_labels()
    ax.legend(handles=line_handles + phase_handles)
    plt.tight_layout()
    plt.show()



In [ ]:
file_dict = {
    "on-the-fly": "./GPUlog_on_OnTheFly.csv",
    "precomputed": "./GPUlog_on_PreComputed.csv"
}

df = load_gpu_logs(file_dict)

In [ ]:
classify_util(df, column="gpu_util_percent", idle_thresh=5, active_thresh=60)

In [ ]:
classify_util(df, column="mem_util_percent", idle_thresh=5, active_thresh=60)

In [ ]:
df_aggr = per_phase_gpu_stats(df)
df_aggr

In [ ]:
per_phase_mem_stats(df)

In [ ]:
df

In [ ]:
per_runtime_stats(df, recording_interval_sec=0.1)

In [ ]:
df_aggr = agg_per_step(df)
df_aggr


In [ ]:
plot_gpu_util_overlay(df_aggr, val_col="gpu_util_median")

In [ ]:
plot_smoothed_util(df_aggr, val_col="gpu_util_median", window=20, device="GPU", calculation='Median')

In [ ]:
plot_smoothed_util(df_aggr, val_col="mem_util_median", window=20, device="Memory", calculation='Median')

In [ ]:
plot_smoothed_util_per_phase(df_aggr, method='on-the-fly', oppacity=1.0, val_col='gpu_util_median', window=20, device="GPU", calculation='Median')

In [ ]:
plot_smoothed_util_per_phase(df_aggr, method='precomputed', oppacity=1, val_col='gpu_util_median', window=20, calculation='Median')